In [48]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [49]:
@tool
def multiply(a:int,b:int):
    """Given two numbers a and b this tool retuns their product"""
    return a*b

In [50]:
print(multiply.invoke({'a':2,'b':3}))

6


In [51]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given two numbers a and b this tool retuns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [52]:
llm=ChatOllama(
    model="qwen3:4b",
    temperature=0.2
)

In [53]:
llm_with_tool1=llm.bind_tools([multiply])

In [54]:
print(llm_with_tool1.invoke("hii, how are you?").content)

The user is greeting me and asking how I am. Since there are no relevant functions provided for handling greetings or responses (the only tool available is for multiplying two integers), I should respond naturally without using any tools.

<response>
Hello! I'm doing well, thank you. How can I assist you today?
</response>


In [55]:
query=HumanMessage('Can you multipy 3 with 5')

In [56]:
messages=[query]

In [57]:
result=llm_with_tool1.invoke(messages)

In [58]:
messages.append(result)

In [59]:
messages

[HumanMessage(content='Can you multipy 3 with 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-21T07:17:35.5613278Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10629003300, 'load_duration': 535887000, 'prompt_eval_count': 151, 'prompt_eval_duration': 54691000, 'eval_count': 355, 'eval_duration': 9997732000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019ee90a-9343-74a3-b9b1-bdef2530b369-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'b86d0047-3a80-4e21-a491-8e4c2e6540ba', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 355, 'total_tokens': 506})]

In [60]:
tool_result=multiply.invoke(result.tool_calls[0])

In [61]:
messages.append(tool_result)

In [62]:
llm_with_tool1.invoke(messages).content

'The product of 3 and 5 is **15**.'

In [63]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """This function fatches the currency factor between a given base currency and a target currency"""
    url=f'https://v6.exchangerate-api.com/v6/0db9f2aecc87790500f09213/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [64]:
llm_with_tool=llm.bind_tools([get_conversion_factor,convert])

In [76]:
messages=[HumanMessage('Whats is the convversion factor between USD to INR and baased on that can you convert 10 usd to inr')]

In [77]:
result=llm_with_tool.invoke(messages)

In [78]:
messages.append(result)

In [79]:
messages

[HumanMessage(content='Whats is the convversion factor between USD to INR and baased on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-21T07:37:49.7405285Z', 'done': True, 'done_reason': 'stop', 'total_duration': 91877213200, 'load_duration': 6357508600, 'prompt_eval_count': 232, 'prompt_eval_duration': 262659000, 'eval_count': 2289, 'eval_duration': 85214189000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019ee91b-dcc5-76a3-8185-ab3d26fa1118-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'dae6cd7c-fa7d-4de9-bd6c-5a98db98e24d', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': '6817d039-e7fd-4d86-a341-f179f27a59e6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 232, 'outp

In [80]:
result.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'dae6cd7c-fa7d-4de9-bd6c-5a98db98e24d',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '6817d039-e7fd-4d86-a341-f179f27a59e6',
  'type': 'tool_call'}]

In [81]:
import json

for tool_call in result.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [82]:
messages

[HumanMessage(content='Whats is the convversion factor between USD to INR and baased on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-21T07:37:49.7405285Z', 'done': True, 'done_reason': 'stop', 'total_duration': 91877213200, 'load_duration': 6357508600, 'prompt_eval_count': 232, 'prompt_eval_duration': 262659000, 'eval_count': 2289, 'eval_duration': 85214189000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019ee91b-dcc5-76a3-8185-ab3d26fa1118-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'dae6cd7c-fa7d-4de9-bd6c-5a98db98e24d', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 94.4113}, 'id': '6817d039-e7fd-4d86-a341-f179f27a59e6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [85]:
llm_with_tool.invoke(messages).content

'The conversion factor between USD and INR is **94.4113**. \n\nUsing this rate, **10 USD** converts to:  \n**944.11 INR** (rounded to two decimal places).'

In [65]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782000001,
 'time_last_update_utc': 'Sun, 21 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782086401,
 'time_next_update_utc': 'Mon, 22 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4113}